# 📖 1. Импорт библиотек


In [ ]:
from bs4 import BeautifulSoup as bs
import requests
import pandas as pd
import re
import json
from datetime import datetime
# импортируем библиотеки 

In [ ]:
url = 'https://telegifter.ru/gifts/collections/' # переменная, содержащая в себе ссылку на сайт для парсинга 

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
}
pages = requests.get(url, headers=headers) # подключаемся
soup = bs(pages.text, 'html.parser') # определяем
print(pages.status_code) # вывод статуса состояния 

In [ ]:
# словарь для будущего датафрейма 
result_list_collection = {
    "collection_id": [],
    "collection_name": [],
    "collection_image_url": [],
    "collection_limit": [],
    "collection_base_price": []
} 

# 2. Парсинг 📤


In [ ]:
# Вспомогательные функции для очистки данных

def clean_price(price_text):
    """Очистка цены от лишних символов"""
    if not price_text or price_text == 'Pre-market':
        return None
    if price_text == '∞':
        return None
    match = re.search(r'[\d.]+', price_text.replace(',', ''))
    if match:
        try:
            return float(match.group())
        except:
            return None
    return None

def clean_limit(limit_text):
    """Очистка лимита"""
    if not limit_text:
        return None
    if '∞' in str(limit_text):
        return -1  # Бесконечный лимит
    digits = re.sub(r'[^\d]', '', str(limit_text))
    if digits:
        return int(digits)
    return None

def generate_id(name):
    """Генерация ID из названия"""
    return re.sub(r'[^a-zA-Z0-9]', '_', name).lower().strip('_')

print('✅ Функции готовы') 

In [ ]:
# 🔍 Сначала сохраним HTML для анализа структуры
with open('telegifter_structure.html', 'w', encoding='utf-8') as f:
    f.write(pages.text)
print('✅ HTML сохранён в telegifter_structure.html')

# Находим все карточки коллекций
# ⚠️ Селекторы нужно адаптировать под реальную структуру сайта
collection_cards = soup.select('.collection-card, .gift-card, .card, .item, [data-collection]')

print(f'📦 Найдено элементов: {len(collection_cards)}')

# Если не нашли по классам, пробуем найти все div с изображениями
if not collection_cards:
    collection_cards = soup.find_all('div', {'class': lambda x: x and ('card' in x or 'item' in x or 'collection' in x)})
    print(f'📦 Альтернативный поиск: {len(collection_cards)} элементов') 

In [ ]:
# Основной цикл парсинга
for card in collection_cards:
    try:
        # Название коллекции
        name_elem = card.select_one('h1, h2, h3, h4, .name, .title, .collection-name')
        collection_name = name_elem.text.strip() if name_elem else ''
        
        if not collection_name or len(collection_name) < 2:
            continue
        
        # Изображение
        img_elem = card.select_one('img')
        collection_image_url = img_elem.get('src', '') if img_elem else ''
        if collection_image_url and not collection_image_url.startswith('http'):
            collection_image_url = 'https://telegifter.ru' + collection_image_url
        
        # Лимит (ищем текст с "ВСЕГО" или числовые значения)
        limit_elem = card.find(string=re.compile(r'ВСЕГО|Total|Supply'))
        if limit_elem:
            collection_limit = clean_limit(limit_elem.parent.text)
        else:
            limit_text = card.select_one('.limit, .total, .supply')
            collection_limit = clean_limit(limit_text.text) if limit_text else None
        
        # Цена
        price_elem = card.find(string=re.compile(r'\$|~\d|TON'))
        if price_elem:
            collection_base_price = clean_price(price_elem.strip())
        else:
            price_text = card.select_one('.price, .base-price, .floor-price')
            collection_base_price = clean_price(price_text.text) if price_text else None
        
        # Генерация ID
        collection_id = generate_id(collection_name)
        
        # Добавляем в словарь
        result_list_collection['collection_id'].append(collection_id)
        result_list_collection['collection_name'].append(collection_name)
        result_list_collection['collection_image_url'].append(collection_image_url)
        result_list_collection['collection_limit'].append(collection_limit)
        result_list_collection['collection_base_price'].append(collection_base_price)
        
    except Exception as e:
        print(f'⚠️ Ошибка обработки карточки: {e}')
        continue

print(f'✅ Парсинг завершён! Собрано {len(result_list_collection["collection_name"])} коллекций') 

In [ ]:
# Просто вывод длины для проверки
print(len(result_list_collection['collection_id']))
print(len(result_list_collection['collection_name']))
print(len(result_list_collection['collection_image_url']))
print(len(result_list_collection['collection_limit']))
print(len(result_list_collection['collection_base_price'])) 

In [ ]:
df = pd.DataFrame(data=result_list_collection) # создаю датафрейм
pd.set_option('max_colwidth', 100) # устанавливаю параметры форматирования
df # вывод 

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
df.to_csv(f'telegifter_collections_{timestamp}.csv', index=False) # Сохраняю в csv формат
print(f'✅ Сохранено в telegifter_collections_{timestamp}.csv') 

In [ ]:
df.info() 

In [ ]:
df.describe().T 

In [ ]:
df.describe(include=object).T 

In [ ]:
df.sample(10) 

In [ ]:
# Дополнительно сохраняем в JSON
with open(f'telegifter_collections_{timestamp}.json', 'w', encoding='utf-8') as f:
    json.dump(result_list_collection, f, ensure_ascii=False, indent=2)
print(f'✅ Сохранено в telegifter_collections_{timestamp}.json') 

In [ ]:
# 📊 Финальная статистика
print(f'\n📈 Статистика:')
print(f'  • Всего коллекций: {len(df)}')
print(f'  • Коллекций с ценой: {df["collection_base_price"].notna().sum()}')
print(f'  • Коллекций с лимитом: {(df["collection_limit"] > 0).sum()}')
if df['collection_base_price'].notna().sum() > 0:
    print(f'  • Средняя цена: {df["collection_base_price"].mean():.2f} TON')
    print(f'  • Мин цена: {df["collection_base_price"].min():.2f} TON')
    print(f'  • Макс цена: {df["collection_base_price"].max():.2f} TON') 